# 03 – Matplotlib: Daten sichtbar machen

**Matplotlib** ist die Standardbibliothek für Grafiken in Python. Fast jede Abbildung in einer wissenschaftlichen Python-Analyse – ob Histogramm, Streudiagramm oder Fieberkurve – entsteht direkt oder indirekt mit Matplotlib.

### Lernziele
Nach diesem Notebook können Sie
- Linien-, Streu-, Balkendiagramme, Histogramme und Boxplots erstellen,
- Achsen beschriften, Titel und Legenden hinzufügen,
- mehrere Grafiken in einer Abbildung kombinieren,
- Abbildungen in Publikationsqualität speichern,
- irreführende Grafiken erkennen.

### Hinweise
- Zellen der Reihe nach mit `Shift + Enter` ausführen.
- ⭐ = wird in der Vorlesung gezeigt, Rest zur Vertiefung.
- Wir verwenden den Datensatz `patienten.csv` aus Notebook 02.

## ⭐ 1. Import

Wir brauchen nur das Untermodul `pyplot`, per Konvention als `plt` abgekürzt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## ⭐ 2. Der erste Plot: eine Fieberkurve

Die einfachste Grafik ist ein Liniendiagramm mit `plt.plot(x, y)`. Eine gute Abbildung braucht aber immer: **Achsenbeschriftungen mit Einheiten** und einen **Titel**.

In [ ]:
tag  = np.arange(1, 8)
temp = np.array([37.1, 38.4, 39.2, 38.9, 38.1, 37.4, 36.9])

plt.plot(tag, temp, marker="o")
plt.axhline(38.0, color="red", linestyle="--", label="Fiebergrenze")   # horizontale Linie
plt.xlabel("Tag nach Aufnahme")
plt.ylabel("Körpertemperatur (°C)")
plt.title("Temperaturverlauf Patient P017")
plt.legend()
plt.show()

Probieren Sie aus: Ändern Sie `marker="o"` zu `marker="s"` oder `"x"`, oder fügen Sie `color="darkgreen"` hinzu.

## ⭐ 3. Figure und Axes – die „richtige“ Schreibweise

Oben haben wir direkt mit `plt.` gezeichnet. Für alles, was über einen einzelnen schnellen Plot hinausgeht, empfiehlt sich die **objektorientierte** Schreibweise:

- **Figure** (`fig`) = das ganze Bild, die „Leinwand“
- **Axes** (`ax`) = ein einzelnes Diagramm darin (eine Figure kann mehrere Axes enthalten)

`plt.subplots()` erzeugt beides auf einmal. Danach zeichnen Sie mit `ax.plot(...)`, `ax.set_xlabel(...)` usw. Beachten Sie: Aus `plt.xlabel` wird `ax.set_xlabel`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))    # Größe in Zoll (Breite, Höhe)

ax.plot(tag, temp, marker="o", label="Temperatur")
ax.axhline(38.0, color="red", linestyle="--", label="Fiebergrenze")
ax.set_xlabel("Tag nach Aufnahme")
ax.set_ylabel("Körpertemperatur (°C)")
ax.set_title("Temperaturverlauf Patient P017")
ax.legend()
plt.show()

## ⭐ 4. Daten laden

Wir laden den Datensatz aus Notebook 02. Falls die Datei nicht gefunden wird (z. B. weil Notebook 02 noch nicht ausgeführt wurde), wird sie automatisch neu erzeugt.

In [ ]:
import os

if not os.path.exists("patienten.csv"):
    # Simulierter Datensatz: 200 Patient*innen einer internistischen Ambulanz
    # (Sie müssen diesen Code nicht im Detail verstehen.)
    import numpy as np
    import pandas as pd

    rng = np.random.default_rng(seed=2024)
    n = 200

    geschlecht = rng.choice(["w", "m"], size=n)
    alter      = rng.integers(18, 90, size=n)
    groesse    = np.where(geschlecht == "m", rng.normal(180, 7, n), rng.normal(167, 6, n)).round(0)
    bmi_wahr   = rng.normal(26.5, 4.5, n).clip(16, 45)
    gewicht    = (bmi_wahr * (groesse / 100) ** 2).round(1)
    raucher    = rng.random(n) < 0.25
    sys_rr     = (100 + 0.5 * alter + 0.9 * (bmi_wahr - 25) + 6 * raucher + rng.normal(0, 12, n)).round(0)
    dia_rr     = (0.5 * sys_rr + rng.normal(15, 6, n)).round(0)
    cholesterin = (165 + 0.7 * alter + rng.normal(0, 30, n)).round(0)
    p_diabetes = 1 / (1 + np.exp(-(-5.5 + 0.045 * alter + 0.15 * (bmi_wahr - 25))))
    diabetes   = rng.random(n) < p_diabetes

    df = pd.DataFrame({
        "patient_id": [f"P{i:03d}" for i in range(1, n + 1)],
        "alter": alter,
        "geschlecht": geschlecht,
        "groesse_cm": groesse,
        "gewicht_kg": gewicht,
        "raucher": raucher,
        "sys_rr": sys_rr,
        "dia_rr": dia_rr,
        "cholesterin": cholesterin,
        "diabetes": diabetes,
    })

    # Ein paar fehlende Werte einbauen – wie im echten Leben
    df.loc[rng.choice(n, 8, replace=False), "cholesterin"] = np.nan
    df.loc[rng.choice(n, 5, replace=False), "gewicht_kg"] = np.nan

    df.to_csv("patienten.csv", index=False)
    print("Datei 'patienten.csv' wurde erstellt.")

df = pd.read_csv("patienten.csv")
df["bmi"] = (df["gewicht_kg"] / (df["groesse_cm"] / 100) ** 2).round(1)
df["hypertonie"] = (df["sys_rr"] >= 140) | (df["dia_rr"] >= 90)
df.head()

## ⭐ 5. Histogramm: Wie ist eine Variable verteilt?

Das Histogramm ist meist die **erste** Grafik, die man sich anschaut. Es zeigt, wie häufig Werte in bestimmten Bereichen (*bins*) vorkommen.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(df["bmi"].dropna(), bins=20, color="steelblue", edgecolor="white")
ax.axvline(25, color="orange", linestyle="--", label="Übergewicht (≥ 25)")
ax.axvline(30, color="red", linestyle="--", label="Adipositas (≥ 30)")
ax.set_xlabel("BMI (kg/m²)")
ax.set_ylabel("Anzahl Patient*innen")
ax.set_title("Verteilung des BMI (n = %d)" % df["bmi"].notna().sum())
ax.legend()
plt.show()

> **Warum `.dropna()`?** Matplotlib kommt mit fehlenden Werten nicht immer gut zurecht. Entfernen Sie `NaN` vor dem Plotten sicherheitshalber.
>
> **Experiment:** Ändern Sie `bins=20` auf `5` und dann auf `60`. Wie verändert sich der Eindruck der Verteilung?

## ⭐ 6. Streudiagramm: Gibt es einen Zusammenhang?

Ein *Scatterplot* stellt zwei kontinuierliche Variablen gegeneinander dar. Jeder Punkt ist eine Patientin bzw. ein Patient.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

raucher       = df[df["raucher"]]
nichtraucher  = df[~df["raucher"]]          # ~ bedeutet "nicht"

ax.scatter(nichtraucher["alter"], nichtraucher["sys_rr"], alpha=0.6, label="Nichtraucher")
ax.scatter(raucher["alter"], raucher["sys_rr"], alpha=0.6, color="crimson", label="Raucher")

ax.set_xlabel("Alter (Jahre)")
ax.set_ylabel("Systolischer Blutdruck (mmHg)")
ax.set_title("Blutdruck in Abhängigkeit vom Alter")
ax.legend()
plt.show()

`alpha` steuert die Transparenz (0 = unsichtbar, 1 = deckend). Das hilft, wenn viele Punkte übereinander liegen.

Möchten Sie eine Regressionsgerade einzeichnen? Mit NumPy geht das in zwei Zeilen:

In [ ]:
steigung, achsenabschnitt = np.polyfit(df["alter"], df["sys_rr"], deg=1)
x_linie = np.array([18, 90])

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["alter"], df["sys_rr"], alpha=0.5)
ax.plot(x_linie, achsenabschnitt + steigung * x_linie, color="black",
        label=f"Regressionsgerade: +{steigung:.2f} mmHg pro Lebensjahr")
ax.set_xlabel("Alter (Jahre)")
ax.set_ylabel("Systolischer Blutdruck (mmHg)")
ax.legend()
plt.show()

## ⭐ 7. Boxplot: Gruppen vergleichen

Boxplots zeigen Median, Quartile und Ausreißer – ideal, um eine Variable zwischen Gruppen zu vergleichen. Wir übergeben eine **Liste** von Datenreihen, eine pro Gruppe.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

daten = [
    df.loc[~df["diabetes"], "cholesterin"].dropna(),
    df.loc[df["diabetes"], "cholesterin"].dropna(),
]
ax.boxplot(daten)
ax.set_xticks([1, 2])
ax.set_xticklabels(["kein Diabetes", "Diabetes"])
ax.set_ylabel("Gesamtcholesterin (mg/dl)")
ax.set_title("Cholesterin nach Diabetes-Status")
plt.show()

> **Lesehilfe Boxplot:** Die Linie in der Box ist der **Median**, die Box reicht vom 1. bis zum 3. Quartil (die mittleren 50 % der Daten). Die „Whisker“ reichen bis zum letzten Wert innerhalb von 1,5 × Box-Höhe, einzelne Punkte darüber hinaus sind potenzielle Ausreißer.

## 8. Balkendiagramm: Häufigkeiten und Anteile

Balkendiagramme eignen sich für kategoriale Daten. Hier kombinieren wir Pandas (`groupby`) und Matplotlib:

In [ ]:
df["altersgruppe"] = pd.cut(df["alter"], bins=[18, 40, 60, 80, 120],
                            labels=["18–39", "40–59", "60–79", "80+"], right=False)
anteil = df.groupby("altersgruppe", observed=True)["hypertonie"].mean() * 100

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(anteil.index.astype(str), anteil.values, color="teal")
ax.set_xlabel("Altersgruppe (Jahre)")
ax.set_ylabel("Anteil mit Hypertonie (%)")
ax.set_title("Hypertonie-Prävalenz nach Altersgruppe")
ax.set_ylim(0, 100)
plt.show()

> **Abkürzung:** Pandas hat eine eingebaute Plot-Funktion, die Matplotlib im Hintergrund nutzt. `anteil.plot(kind="bar")` liefert ein ähnliches Ergebnis. Für schnelle Erkundung praktisch, für Feinschliff greift man auf Matplotlib zurück.

## ⭐ 9. Mehrere Grafiken in einer Abbildung

Mit `plt.subplots(zeilen, spalten)` erhalten Sie ein ganzes Raster von Axes. `axes` ist dann ein Array, das Sie wie in NumPy indizieren: `axes[0, 1]` = erste Zeile, zweite Spalte.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].hist(df["alter"], bins=15, color="slategray", edgecolor="white")
axes[0, 0].set_title("Altersverteilung")
axes[0, 0].set_xlabel("Alter (Jahre)")

axes[0, 1].scatter(df["bmi"], df["sys_rr"], alpha=0.5)
axes[0, 1].set_title("BMI vs. systolischer Blutdruck")
axes[0, 1].set_xlabel("BMI (kg/m²)")
axes[0, 1].set_ylabel("RR sys (mmHg)")

zaehlung = df["geschlecht"].value_counts()
axes[1, 0].bar(zaehlung.index, zaehlung.values, color=["orchid", "cornflowerblue"])
axes[1, 0].set_title("Geschlechterverteilung")
axes[1, 0].set_ylabel("Anzahl")

axes[1, 1].boxplot([df.loc[df["geschlecht"] == "w", "sys_rr"],
                    df.loc[df["geschlecht"] == "m", "sys_rr"]])
axes[1, 1].set_xticks([1, 2])
axes[1, 1].set_xticklabels(["w", "m"])
axes[1, 1].set_title("Blutdruck nach Geschlecht")
axes[1, 1].set_ylabel("RR sys (mmHg)")

fig.suptitle("Übersicht Ambulanz-Kohorte (n = 200)", fontsize=14)
fig.tight_layout()     # verhindert überlappende Beschriftungen
plt.show()

## ⭐ 10. Abbildungen speichern

Mit `fig.savefig` speichern Sie eine Abbildung als Datei. Für Doktorarbeit oder Paper:
- `dpi=300` für hohe Auflösung (Standard bei Journals),
- `bbox_inches="tight"` schneidet überflüssigen weißen Rand ab,
- Formate: `.png` (Pixel), `.pdf` oder `.svg` (Vektorgrafik, beliebig skalierbar).

**Wichtig:** `savefig` muss **vor** `plt.show()` stehen, danach ist die Figure „verbraucht“.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["sys_rr"], bins=20, color="steelblue", edgecolor="white")
ax.set_xlabel("Systolischer Blutdruck (mmHg)")
ax.set_ylabel("Anzahl")

fig.savefig("blutdruck_histogramm.png", dpi=300, bbox_inches="tight")
plt.show()
print("Gespeichert als blutdruck_histogramm.png")

## 11. Vorsicht: Wie Grafiken täuschen können

Dieselben Daten können sehr unterschiedlich wirken. Ein Klassiker in Werbung und leider auch in der Wissenschaft: die **abgeschnittene y-Achse**.

In [ ]:
gruppen = ["Placebo", "Medikament"]
werte   = [142, 138]       # mittlerer systolischer Blutdruck nach 12 Wochen

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(gruppen, werte, color=["gray", "seagreen"])
ax1.set_ylim(136, 143)
ax1.set_title("Irreführend: y-Achse beginnt bei 136")
ax1.set_ylabel("RR sys (mmHg)")

ax2.bar(gruppen, werte, color=["gray", "seagreen"])
ax2.set_ylim(0, 160)
ax2.set_title("Ehrlich: y-Achse beginnt bei 0")
ax2.set_ylabel("RR sys (mmHg)")

fig.tight_layout()
plt.show()

Links sieht der Effekt dramatisch aus, rechts sieht man: Es sind 4 mmHg. **Bei Balkendiagrammen sollte die y-Achse immer bei 0 beginnen**, denn die Länge des Balkens codiert den Wert.

### ✅ Checkliste für gute Abbildungen
- Achsen beschriftet – **mit Einheiten**
- Aussagekräftiger Titel bzw. Bildunterschrift
- Legende, wenn mehrere Gruppen gezeigt werden
- Stichprobengröße (n) angegeben
- Farben auch für Menschen mit Rot-Grün-Schwäche unterscheidbar
- Keine abgeschnittenen Achsen bei Balkendiagrammen
- Lieber Einzelwerte bzw. Verteilungen zeigen (Boxplot, Streudiagramm) als nur Mittelwert-Balken

---
## 📝 Übungsaufgaben

### Aufgabe 1: Blutzucker-Tagesprofil
Eine Patientin mit Diabetes hat folgendes Tagesprofil (Uhrzeit → Glukose in mg/dl):
`7 → 135, 9 → 210, 12 → 160, 14 → 230, 18 → 175, 20 → 245, 23 → 150`.

Erstellen Sie ein Liniendiagramm mit Markern und zeichnen Sie den Zielbereich (70–180 mg/dl) als grün schattierte Fläche ein. *Tipp:* `ax.axhspan(70, 180, color="green", alpha=0.15)`

In [ ]:
# Ihre Lösung hier

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
uhrzeit = [7, 9, 12, 14, 18, 20, 23]
glukose = [135, 210, 160, 230, 175, 245, 150]

fig, ax = plt.subplots(figsize=(8, 4))
ax.axhspan(70, 180, color="green", alpha=0.15, label="Zielbereich")
ax.plot(uhrzeit, glukose, marker="o", color="darkblue", label="Glukose")
ax.set_xlabel("Uhrzeit (h)")
ax.set_ylabel("Glukose (mg/dl)")
ax.set_title("Blutzucker-Tagesprofil")
ax.set_xticks(uhrzeit)
ax.legend()
plt.show()
```
</details>

### Aufgabe 2: Gewicht vs. Größe
Erstellen Sie ein Streudiagramm von Körpergröße (x) gegen Körpergewicht (y), mit Frauen und Männern in unterschiedlichen Farben und einer Legende. Vergessen Sie nicht die Einheiten!

In [ ]:
# Ihre Lösung hier

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
fig, ax = plt.subplots(figsize=(7, 5))
for g, farbe, name in [("w", "orchid", "Frauen"), ("m", "cornflowerblue", "Männer")]:
    teil = df[df["geschlecht"] == g]
    ax.scatter(teil["groesse_cm"], teil["gewicht_kg"], alpha=0.6, color=farbe, label=name)
ax.set_xlabel("Körpergröße (cm)")
ax.set_ylabel("Körpergewicht (kg)")
ax.set_title("Größe und Gewicht nach Geschlecht")
ax.legend()
plt.show()
```
Die `for`-Schleife spart doppelten Code: Sie läuft einmal für Frauen und einmal für Männer.
</details>

### Aufgabe 3: Eigene Abbildung
Erstellen Sie eine Abbildung mit **zwei nebeneinanderliegenden** Grafiken (`plt.subplots(1, 2)`):
- links: Histogramm des Cholesterins,
- rechts: Boxplot des systolischen Blutdrucks für Raucher vs. Nichtraucher.

Speichern Sie die Abbildung als PDF.

In [ ]:
# Ihre Lösung hier

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.hist(df["cholesterin"].dropna(), bins=20, color="goldenrod", edgecolor="white")
ax1.set_xlabel("Gesamtcholesterin (mg/dl)")
ax1.set_ylabel("Anzahl")
ax1.set_title("Cholesterin")

ax2.boxplot([df.loc[~df["raucher"], "sys_rr"], df.loc[df["raucher"], "sys_rr"]])
ax2.set_xticks([1, 2])
ax2.set_xticklabels(["Nichtraucher", "Raucher"])
ax2.set_ylabel("RR sys (mmHg)")
ax2.set_title("Blutdruck nach Rauchstatus")

fig.tight_layout()
fig.savefig("meine_abbildung.pdf", bbox_inches="tight")
plt.show()
```
</details>

---
## 📌 Zusammenfassung

| Diagrammtyp | Wofür? | Code |
|---|---|---|
| Linie | Verläufe über Zeit | `ax.plot(x, y)` |
| Histogramm | Verteilung einer Variable | `ax.hist(x, bins=20)` |
| Streudiagramm | Zusammenhang zweier Variablen | `ax.scatter(x, y)` |
| Boxplot | Verteilung in Gruppen vergleichen | `ax.boxplot([gruppe1, gruppe2])` |
| Balken | Häufigkeiten, Anteile | `ax.bar(kategorien, werte)` |

| Beschriften & Gestalten | Code |
|---|---|
| Figure + Axes | `fig, ax = plt.subplots(figsize=(7, 4))` |
| Raster | `fig, axes = plt.subplots(2, 2)` |
| Achsen & Titel | `ax.set_xlabel(...)`, `ax.set_ylabel(...)`, `ax.set_title(...)` |
| Referenzlinien | `ax.axhline(y)`, `ax.axvline(x)`, `ax.axhspan(y1, y2)` |
| Legende | `label="..."` beim Zeichnen + `ax.legend()` |
| Achsenbereich | `ax.set_ylim(0, 100)` |
| Speichern | `fig.savefig("abb.png", dpi=300, bbox_inches="tight")` |

**Weiter geht’s mit Notebook 04 – scikit-learn**, wo wir ein erstes Vorhersagemodell trainieren.